# FedKDL Google Colab Experiments Runner
Notebook này được tinh chỉnh riêng cho môi trường Google Colab. Nó sẽ tự động kết nối với Google Drive của bạn để lưu kết quả, tránh việc mất dữ liệu khi session bị ngắt.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1. Tải Source Code từ GitHub
Clone repo FedKDL mới nhất từ GitHub và di chuyển vào thư mục làm việc.

In [ ]:
!git clone https://github.com/ngnam1104/FedKDL.git
import os
os.chdir('/content/FedKDL')
print('Current Working Directory:', os.getcwd())

### 2. Cài đặt thư viện Cài đặt thư viện và Khởi tạo đường dẫn


In [ ]:
!pip install ultralytics pandas pyyaml torch torchvision
import os, sys
sys.path.append(os.getcwd())

!mkdir -p /content/FedKDL/datasets/URPC2020
# [QUAN TRỌNG] Hãy sửa đường dẫn /content/drive/MyDrive/... bên dưới trỏ tới đúng thư mục dataset URPC2020 trên Google Drive của bạn
!ln -s /content/drive/MyDrive/URPC2020 /content/FedKDL/datasets/URPC2020/URPC2020



### 3. Tạo Topologies và Data Partitions

In [ ]:
!python utils/generate_all_envs.py --n 30 --dataset URPC --m-relays 5 --alphas 1.0

### 4.1. Pre-train Centralized Full-Finetune (Upper Bound)
Chạy Centralized Full Finetune trên toàn bộ dữ liệu để có baseline Upper Bound cho cấu trúc gốc không dùng LoRA.

In [ ]:
!python scripts/fedkdl/train_student_warmup.py --mode centralized_full
# Nếu bị ngắt giữa chừng, đổi thành lệnh dưới để chạy tiếp:
# !python scripts/fedkdl/train_student_warmup.py --mode centralized_full --resume

### 4.2. Pre-train Centralized LoRA
Chạy Centralized LoRA trên toàn bộ dữ liệu để so sánh hiệu năng trực tiếp với Full Finetune.

In [ ]:
!python scripts/fedkdl/train_student_warmup.py --mode centralized_lora
# Nếu bị ngắt giữa chừng, đổi thành lệnh dưới để chạy tiếp:
# !python scripts/fedkdl/train_student_warmup.py --mode centralized_lora --resume

### 5. Khởi chạy thuật toán FedKDL (Main Baseline)
**Giải đáp về Log:** Trong lệnh dưới, toán tử `2>&1 | tee ...` sẽ vừa in log ra giao diện Kaggle Notebook để bạn xem trực tiếp, VỪA ghi song song toàn bộ log vào file `.log`. Kể cả khi Notebook của bạn bị reload/treo trình duyệt, file log vật lý trên hệ thống Kaggle vẫn được lưu bình thường không sót chữ nào!

In [ ]:
!pip uninstall -y ray

In [ ]:
!mkdir -p results/logs_kdl results/train_logs/kdl

!python main_trainer_od.py \
    --topo environments/2d/topo/N_30/topo_N30_seed1104.pkl \
    --data environments/2d/data/URPC/N_30/data_N30_URPC_a1p0_seed1104.pkl \
    --baseline fedkdl \
    --rounds 60 \
    --out-dir results/logs_kdl \
    --log-dir results/train_logs/kdl \
    2>&1 | tee results/train_logs/kdl/kaggle_fedkdl_raw.log

### 6. Khởi chạy các Kịch bản So sánh (Ablation)
Bạn có thể đổi `baseline` thành `fedkdl_selective`, `fedprox_kdl`, `fedkd` tùy nhu cầu để chạy các thử nghiệm khác.

In [ ]:
!python main_trainer_od.py \
    --topo environments/2d/topo/N_30/topo_N30_seed1104.pkl \
    --data environments/2d/data/URPC/N_30/data_N30_URPC_a1p0_seed1104.pkl \
    --baseline fedkd \
    --rounds 60 \
    --out-dir results/logs_kdl \
    --log-dir results/train_logs/kdl \
    2>&1 | tee results/train_logs/kdl/kaggle_fedkd_raw.log

### 7. Lưu Toàn Bộ Kết Quả Về Google Drive
Chạy block này để nén và copy toàn bộ logs, weights, kết quả mAP sang Google Drive của bạn.

In [ ]:
import os
os.chdir('/content/FedKDL')

# Tạo thư mục backup trên Drive
!mkdir -p /content/drive/MyDrive/FedKDL_Backup

# Copy folder results (chứa loss, log, mAP csv)
!cp -r results /content/drive/MyDrive/FedKDL_Backup/results

# Nén và copy folder runs (chứa weights của Ultralytics)
!zip -r runs_backup.zip runs/
!cp runs_backup.zip /content/drive/MyDrive/FedKDL_Backup/

print('Đã lưu toàn bộ kết quả an toàn lên Google Drive!')